# **Gated Recurrent Units (GRU)**
The core idea behind GRUs is to use gating mechanisms to selectively update the hidden state at each time step allowing them to remember important information while discarding irrelevant details. GRUs aim to simplify the LSTM architecture by merging some of its components and focusing on just two main gates: the update gate and the reset gate.

GRU consists of two main gates:

- **Update Gate:** This gate decides how much information from previous hidden state should be retained for the next time step.
- **Reset Gate:** This gate determines how much of the past hidden state should be forgotten.

### **1. Importing Libraries**

We will import the necessary libraries for implementing our GRU model such as numpy, pandas, MinMaxScaler, TensorFlow and Adam.

In [13]:
import numpy as np
import pandas as pd
from keras.layers import Input
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam

### **2. Loading the Dataset**

The dataset we're using is a time-series dataset containing daily temperature data i.e forecasting dataset. It spans 8,000 days starting from January 1, 2010.

- **pd.read_csv():** Reads a CSV file into a pandas DataFrame. Here, we are assuming that the dataset has a Date column which is set as the index of the DataFrame.
- **parse_dates=['Date']:** Ensures that the 'Date' column is automatically converted into datetime format.

In [14]:
df = pd.read_csv('/Users/tannutiwari/Downloads/Projects/Jupyter/data.csv', parse_dates=['Date'], index_col='Date')
print(df.head())

            Temperature
Date                   
2010-01-01    27.483571
2010-01-02    24.308678
2010-01-03    28.238443
2010-01-04    32.615149
2010-01-05    23.829233


### **3. Preprocessing the Data**

We will scale our data to ensure all features have equal weight and avoid any bias. In this example, we will use MinMaxScaler, which scales the data to a range between 0 and 1. Proper scaling is important because neural networks tend to perform better when input features are normalized.

**MinMaxScaler**
- It is used to scale data into a fixed range, usually 0 to 1

In [15]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df.values)

### **4. Preparing Data for GRU**

We will define a function to prepare our data for training our model.

- **create_dataset():** Prepares the dataset for time-series forecasting. It creates sliding windows of time_step length to predict the next time step.
- **X.reshape():** Reshapes the input data to fit the expected shape for the GRU which is 3D: i.e samples, time steps and features.

In [16]:
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)


time_step = 100
X, y = create_dataset(scaled_data, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)

### **5. Building the GRU Model**

We will define our GRU model with the following components:

- **GRU(units=50):** Adds a GRU layer with 50 units (neurons).
- **return_sequences=True:** Ensures that the GRU layer returns the entire sequence (required for stacking multiple GRU layers).
- **Dense(units=1):** The output layer which predicts a single value for the next time step.
- **Adam():** An adaptive optimizer commonly used in deep learning.



In [17]:
model = Sequential([
    Input(shape=(X.shape[1], 1)),
    GRU(units=50, return_sequences=True),
    GRU(units=50),
    Dense(units=1)
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='mean_squared_error')

### **6. Training the Model**

model.fit() trains the model on the prepared dataset. The epochs=10 specifies the number of iterations over the entire dataset, and batch_size=32 defines the number of samples per batch.

In [18]:
model.fit(X, y, epochs=10, batch_size=32)

Epoch 1/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - loss: 0.0237
Epoch 2/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - loss: 0.0179
Epoch 3/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - loss: 0.0179
Epoch 4/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - loss: 0.0178
Epoch 5/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - loss: 0.0178
Epoch 6/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - loss: 0.0179
Epoch 7/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - loss: 0.0178
Epoch 8/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - loss: 0.0178
Epoch 9/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - loss: 0.0177
Epoch 10/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - loss: 0.0178


### **7. Making Predictions**

We will be now making predictions using our trained GRU model.

- **Input Sequence:** The code takes the last 100 temperature values from the dataset (scaled_data[-time_step:]) as an input sequence.
- **Reshaping the Input Sequence:** The input sequence is reshaped into the shape (1, time_step, 1) because the GRU model expects a 3D input: [samples, time_steps, features]. Here samples=1 because we are making one prediction, time_steps=100 (the length of the input sequence) and features=1 because we are predicting only the temperature value.
- **model.predict():** Uses the trained model to predict future values based on the input data.

In [19]:
input_sequence = scaled_data[-time_step:].reshape(1, time_step, 1)
predicted_values = model.predict(input_sequence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step


### **8. Inverse Transforming the Predictions**

Inverse Transforming the Predictions refers to the process of converting the scaled (normalized) predictions back to their original scale.

- **scaler.inverse_transform():** Converts the normalized predictions back to their original scale.

In [20]:
predicted_values = scaler.inverse_transform(predicted_values)
print(
    f"The predicted temperature for the next day is: {predicted_values[0][0]:.2f}°C")

The predicted temperature for the next day is: 24.26°C


# **Seq2Seq Model**
Sequence‑to‑Sequence (Seq2Seq) models are neural networks designed to transform one sequence into another, even when the input and output lengths differ and are built using encoder‑decoder architecture. A Sequence-to-Sequence (Seq2Seq) model consists of two primary phases: encoding the input sequence and decoding it into an output sequence.

## **1. Encoding the Input Sequence**

- The encoder processes the input sequence token by token, updating its internal state at each step.
- After processing the entire sequence, the encoder produces a context vector i.e a fixed-length representation summarizing the important information from the input.

## **2. Decoding the Output Sequence**

The decoder takes the context vector and generates the output sequence one token at a time. For example, in machine translation:

Input: "I am learning"
Output: "Je suis apprenant"
Each token is predicted based on the context vector and previously generated tokens.

## **3. Teacher Forcing**

During training, teacher forcing is commonly used. Instead of feeding the decoder’s own previous prediction as the next input, the actual target token from the training data is provided.

Benefits:

- Accelerates training
- Reduces error propagation
Teacher forcing is used only during training and not during inference, where the model relies on its own previous predictions.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


**Note:** Choose the installation method that best fits your environment. The conda method is usually most reliable in Anaconda/Jupyter environments. If you need GPU support, use the CUDA version in Method 3.

### **Step 2: Encoder**

We will define:

- Each input token is converted to a dense vector (embedding).
- The GRU processes the sequence one token at a time, updating its hidden state.
- The final hidden state is returned as the context vector, summarizing the input sequence.

In [2]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)
        return hidden

### **Step 3: Decoder**

We will define the decoder:

- Takes the current input token and converts it to an embedding.
- GRU uses the previous hidden state (or context vector initially) to compute the new hidden state.
- The output is passed through a linear layer to get predicted token probabilities.

In [3]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden):
        input = input.unsqueeze(0)
        embedded = self.embedding(input)
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc(output.squeeze(0))
        return prediction, hidden

### **Step 4: Seq2Seq Model with Teacher Forcing**

- Batch size & vocab size: extracted from input and decoder.
- Encoding: input sequence → encoder → context vector (hidden).
- Start token: initialize decoder with token 0.
- Loop over max_len:
- Decoder predicts next token.
- top1 → token with max probability.
- Append top1 to outputs.
- Teacher forcing: sometimes feed true target token instead of prediction.
- Return predictions: concatenated sequence of token IDs.

In [4]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg=None, max_len=10, teacher_forcing_ratio=0.5):
        batch_size = src.shape[1]
        trg_vocab_size = self.decoder.fc.out_features
        outputs = []

        hidden = self.encoder(src)

        input = torch.zeros(batch_size, dtype=torch.long).to(self.device)

        for t in range(max_len):
            output, hidden = self.decoder(input, hidden)
            top1 = output.argmax(1)
            outputs.append(top1.unsqueeze(0))

            if trg is not None and t < trg.shape[0] and torch.rand(1).item() < teacher_forcing_ratio:
                input = trg[t]
            else:
                input = top1

        outputs = torch.cat(outputs, dim=0)
        return outputs

## **Step 5: Usage Example with Outputs**

Test with example,

- src: random input token IDs.
- trg: random target token IDs (used for teacher forcing).
- outputs: predicted token IDs for each sequence.
- .T: transpose to show batch sequences as rows.

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = 10
EMB_DIM = 8
HID_DIM = 16
SEQ_LEN = 5
BATCH_SIZE = 2

enc = Encoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
dec = Decoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
model = Seq2Seq(enc, dec, device).to(device)

src = torch.randint(1, VOCAB_SIZE, (SEQ_LEN, BATCH_SIZE)).to(device)
trg = torch.randint(1, VOCAB_SIZE, (SEQ_LEN, BATCH_SIZE)).to(device)

outputs = model(src, trg, max_len=SEQ_LEN, teacher_forcing_ratio=0.7)

print("Source sequence (input tokens):")
print(src.T)
print("\nTarget sequence (true tokens):")
print(trg.T)
print("\nPredicted sequence (model output tokens):")
print(outputs.T)

Source sequence (input tokens):
tensor([[5, 7, 2, 1, 8],
        [4, 1, 1, 3, 1]])

Target sequence (true tokens):
tensor([[8, 3, 1, 4, 7],
        [6, 3, 5, 8, 5]])

Predicted sequence (model output tokens):
tensor([[1, 0, 0, 0, 4],
        [1, 4, 4, 4, 0]])
